# 04 - Model Interpretability

This notebook demonstrates:
- SHAP value computation and visualization
- Feature importance ranking
- Partial Dependence Plots
- Identifying key hydrological drivers

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import joblib
import matplotlib.pyplot as plt

from src.data.loader import CAMELSIndDataLoader
from src.data.preprocessor import StreamflowPreprocessor
from src.features.engineer import FeatureEngineer
from src.models.interpretability import SHAPExplainer
from src.utils.helpers import set_seed

set_seed(42)
%matplotlib inline

In [ ]:
# Load data and model
loader = CAMELSIndDataLoader('../data/raw')
data = loader.load_all_catchments()
engineer = FeatureEngineer()
data = engineer.transform(data)
feature_names = engineer.get_feature_names(data)

preprocessor = StreamflowPreprocessor()
X_train, X_test, y_train, y_test = preprocessor.fit_transform(data)

# Load or train model
try:
    saved = joblib.load('../outputs/models/best_model.joblib')
    model = saved['model']
    print(f'Loaded model: {saved["model_name"]}')
except FileNotFoundError:
    from sklearn.ensemble import RandomForestRegressor
    model = RandomForestRegressor(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)
    print('Trained new RandomForest model')

In [ ]:
# SHAP analysis
feat_names = preprocessor.feature_columns or [f'f_{i}' for i in range(X_test.shape[1])]
explainer = SHAPExplainer(model, feat_names, output_dir='../outputs/figures')

shap_values = explainer.compute_shap_values(X_test, sample_size=300)
importance_df = explainer.plot_feature_importance(top_n=20)

print('Top 10 most important features:')
print(importance_df.head(10).to_string(index=False))

In [ ]:
# SHAP summary plot
explainer.plot_summary(X_test, save=True)
print('SHAP summary plot saved')

In [ ]:
# Dependence plots for top features
top_3 = importance_df.head(3)['feature'].tolist()
for feat in top_3:
    explainer.plot_dependence(feat, X_test)
    print(f'Dependence plot for {feat} saved')